# Data Preparataion

## 1. Imports

In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

## 2. Data retrieval

### 2.1 Load IR-Plag-Dataset

In [2]:
ir_plag_root = "../Dataset/IR-Plag-Dataset"

rows = []

# Reads JAVA file to save in on an attribute of the dataframe
def read_code(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()

# Walks through the dataset to read all the files and save them in a dataframe with their respective label
for case_dir in os.listdir(ir_plag_root):
    case_path = os.path.join(ir_plag_root, case_dir)
    if not os.path.isdir(case_path):
        continue

    original_dir = os.path.join(case_path, "original")
    if not os.path.isdir(original_dir):
        continue

    original_files = os.listdir(original_dir)
    if not original_files:
        continue

    original_code = read_code(os.path.join(original_dir, original_files[0]))

    for label_folder, label in [("plagiarized", 1), ("non-plagiarized", 0)]:
        label_dir = os.path.join(case_path, label_folder)
        if not os.path.isdir(label_dir):
            continue

        # Traverse subfolders to extract files
        for current_path, dirs, files in os.walk(label_dir):
            for file in files:
                candidate_code = read_code(os.path.join(current_path, file))
                rows.append({
                    "code1": original_code,
                    "code2": candidate_code,
                    "label": label
                })

df_ir_plag = pd.DataFrame(rows)

df_ir_plag.head()

,code1,code2,label
0,\npublic class T1 {\n\tpublic static void main...,"/*\n * To change this license header, choose L...",1
1,\npublic class T1 {\n\tpublic static void main...,public class Main {\n\n public static void ...,1
2,\npublic class T1 {\n\tpublic static void main...,"\n\n\n\n/*\n * To change this license header, ...",1
3,\npublic class T1 {\n\tpublic static void main...,\npublic class T1 {\n\tpublic static void main...,1
4,\npublic class T1 {\n\tpublic static void main...,class HelloWorld\n{\n\tpublic static void main...,1


In [3]:
df_ir_plag['label'].value_counts()

label
1    355
0    105
Name: count, dtype: int64

### 2.2 Load conplag_version_2 dataset

In [4]:
conplag_path = "../Dataset/conplag_version_2/versions"

version_folders = [
    "bplag_version_1",
    "bplag_version_2",
    "version_1",
    "version_2"
]

labels = pd.read_csv(conplag_path + "/labels.csv")

train_pairs = pd.read_csv(
    conplag_path + "/train_pairs.csv",
    header=None,
    names=["pair"]
)

test_pairs = pd.read_csv(
    conplag_path + "/test_pairs.csv",
    header=None,
    names=["pair"]
)

# Separate pair into submission 1 and submission 2
# e.g. "00af3420_5449d33c" -> sub1: "00af3420", sub2: "5449d33c"
train_pairs[["sub1", "sub2"]] = train_pairs["pair"].str.split("_", expand=True)
test_pairs[["sub1", "sub2"]] = test_pairs["pair"].str.split("_", expand=True)

# Add labels
# e.g. for pair "00af3420_5449d33c"
#     sub1      sub2        label
# 0   00af3420  5449d33c    1
train_df = train_pairs.merge(labels, on=["sub1", "sub2"], how="left")
test_df = test_pairs.merge(labels, on=["sub1", "sub2"], how="left")

train_df = train_df.rename(columns={"verdict": "label"})
test_df = test_df.rename(columns={"verdict": "label"})

# Read code files for each pair and add them to the dataframe
def read_code(version, pair, sub):
    folder = os.path.join(conplag_path, version, pair, sub)

    if not os.path.exists(folder):
        return None
    
    file_path = os.path.join(folder, sub + ".java")

    if not os.path.exists(file_path):
        return None

    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()
    
conplag_train_dfs = []
conplag_test_dfs = []

for version in version_folders:
    train_temp = train_df.copy()
    test_temp = test_df.copy()

    train_temp["version"] = version
    test_temp["version"] = version

    # Apply the read_code function to each row of the dataframe to create new columns "code1" and "code2" for train and test dataframes
    train_temp["code1"] = train_temp.apply(
            lambda row: read_code(version, row["pair"], row["sub1"]),
            axis=1
        )
    train_temp["code2"] = train_temp.apply(lambda row: read_code(version, row["pair"], row["sub2"]), axis=1)
    test_temp["code1"] = test_temp.apply(lambda row: read_code(version, row["pair"], row["sub1"]), axis=1)
    test_temp["code2"] = test_temp.apply(lambda row: read_code(version, row["pair"], row["sub2"]), axis=1)

    conplag_train_dfs.append(train_temp)
    conplag_test_dfs.append(test_temp)


conplag_train_df = pd.concat(conplag_train_dfs, ignore_index=True)
conplag_test_df = pd.concat(conplag_test_dfs, ignore_index=True)

# keeps only essential columns for training and testing the model
conplag_train_df = conplag_train_df[["code1", "code2", "label"]]
conplag_test_df = conplag_test_df[["code1", "code2", "label"]]

# Data cleaning
conplag_train_df = conplag_train_df.dropna()
conplag_test_df = conplag_test_df.dropna()

conplag_test_df.head()

,code1,code2,label
0,import java.util.*;\n\npublic class Soltion{\n...,import java.util.*;\n\npublic class mentor1 {\...,0
1,import java.io.*;\nimport java.util.*;\n\npubl...,import java.io.*;\nimport java.util.*;\n\npubl...,1
2,import java.util.*;\nimport java.io.*;\npublic...,import java.io.*;\nimport java.util.*;\npublic...,0
3,import java.util.*;\nimport java.io.*;\n\n\npu...,import java.io.*;\nimport java.util.*;\npublic...,0
4,import java.io.*;\nimport java.util.*;\npublic...,import java.io.*;\nimport java.util.*;\n\npubl...,0


In [5]:
conplag_test_df['label'].value_counts()

label
0    990
1    372
Name: count, dtype: int64

### 2.3 Load AI detection dataset

In [6]:
ai_detect_root = "../Dataset/ai_detection"
json_file = os.path.join(ai_detect_root, "java_dataset.jsonl")

# Load dataset
df = pd.read_json(json_file, lines=True)

# Human samples
df_human = pd.DataFrame({
    "code": df["human_code"],
    "label": 0
})

# AI samples (ChatGPT)
df_ai = pd.DataFrame({
    "code": df["chatgpt_code"],
    "label": 1
})

# Join datasets
df_ai_detection = pd.concat(
    [df_human, df_ai],
    ignore_index=True
)

df_ai_detection.head()

,code,label
0,private OptionKindAndValue readKindAndValue() ...,0
1,public <TContinuationResult> Task<TContinuatio...,0
2,public static String getUserAgent() {\n ...,0
3,public String translatePathToLocation( String ...,0
4,"public void process( T image1 , T image2 )\n\t...",0


In [7]:
df_ai_detection['label'].value_counts()

label
0    221796
1    221796
Name: count, dtype: int64

### 2.4 Load id2source dataset

In [10]:
id2sourcecode_path = "../Dataset/id2sourcecode"

# Obtain pair of positive cases (plagiarized) and negative cases (non-plagiarized) from the dataset
clone = pd.read_csv(id2sourcecode_path + "/clone_pairs.csv")
nonclone = pd.read_csv(id2sourcecode_path + "/nonclone_pairs.csv")

# Keep only first two columns
clone = clone.iloc[:, 0:2].copy()
nonclone = nonclone.iloc[:, 0:2].copy()

# Rename columns
clone.columns = ["id_1", "id_2"]
nonclone.columns = ["id_1", "id_2"]

# Add labels
clone["label"] = 1
nonclone["label"] = 0

df_pairs = pd.concat(
    [clone, nonclone],
    ignore_index=True
)

print(df_pairs.head())
print(df_pairs["label"].value_counts())

# Build lookup dictionary reading each unique file only once
id_set = set(df_pairs["id_1"]).union(set(df_pairs["id_2"]))

id2code = {}
for file_id in id_set:
    file_path = os.path.join(id2sourcecode_path, str(file_id) + ".java")
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        id2code[file_id] = f.read()

print(f"Loaded {len(id2code)} unique files")

# Map codes from the dictionary instead of re-reading from disk
df_pairs["code1"] = df_pairs["id_1"].map(id2code)
df_pairs["code2"] = df_pairs["id_2"].map(id2code)

# Keep only essential columns for training and testing the model
df_plagiarism_detection = df_pairs[["id_1", "id_2", "code1", "code2", "label"]].copy()

       id_1      id_2  label
0   8001867  23594635      1
1  15537156  23594635      1
2  20619879  23594635      1
3  16499420  23594635      1
4  20601755  23594635      1
label
0    279032
1    269999
Name: count, dtype: int64
Loaded 73135 unique files


In [11]:
df_plagiarism_detection.head()

,id_1,id_2,code1,code2,label
0,8001867,23594635,private void nioBuild() {\n try {\n...,private void copyFileToPhotoFolder(File ph...,1
1,15537156,23594635,"private void copy(String inputPath, String...",private void copyFileToPhotoFolder(File ph...,1
2,20619879,23594635,public void copyLogic() {\n if (get...,private void copyFileToPhotoFolder(File ph...,1
3,16499420,23594635,"private void saveFile(InputStream in, Stri...",private void copyFileToPhotoFolder(File ph...,1
4,20601755,23594635,public static File copyFile(File file) {\n...,private void copyFileToPhotoFolder(File ph...,1


In [12]:
df_plagiarism_detection["label"].value_counts()

label
0    279032
1    269999
Name: count, dtype: int64

## 3. Data verification

### 3.1 Verification for IR-Plag-Dataset

In [18]:
if df_ir_plag.isnull().values.any():
    print("There are null values in the dataset, data cleaning is required.")
else:
    print("There are no null values in the IR-PLAG dataset, data is clean.")

There are no null values in the IR-PLAG dataset, data is clean.


In [14]:
initial_count_ir_plag = len(df_ir_plag)

df_ir_plag = df_ir_plag[df_ir_plag["code1"].str.strip().str.len() > 5]
df_ir_plag = df_ir_plag[df_ir_plag["code2"].str.strip().str.len() > 5]

after_empty_ir_plag = len(df_ir_plag)

print(f"Removed {initial_count_ir_plag - after_empty_ir_plag} pairs with empty code")

Removed 0 pairs with empty code


In [15]:
df_ir_plag = df_ir_plag.drop_duplicates(subset=["code1", "code2"])

after_duplicates_ir_plag = len(df_ir_plag)

print(f"Removed {after_empty_ir_plag - after_duplicates_ir_plag} duplicated pairs")

Removed 13 duplicated pairs


In [16]:
print(f"Final pairs: {after_duplicates_ir_plag}")

Final pairs: 447


### 3.2 Verification for conplag_version_2 dataset

In [17]:
if conplag_train_df.isnull().values.any() or conplag_test_df.isnull().values.any():
    print("There are null values in the ConPlag dataset, data cleaning is required.")
else:
    print("There are no null values in the ConPlag dataset, data is clean.")

There are no null values in the ConPlag dataset, data is clean.


In [19]:
initial_count_conplag_train = len(conplag_train_df)

conplag_train_df = conplag_train_df[conplag_train_df["code1"].str.strip().str.len() > 5]
conplag_train_df = conplag_train_df[conplag_train_df["code2"].str.strip().str.len() > 5]

after_empty_conplag_train = len(conplag_train_df)

print(f"Removed {initial_count_conplag_train - after_empty_conplag_train} pairs with empty code")

Removed 0 pairs with empty code


In [20]:
initial_count_conplag_test = len(conplag_test_df)

conplag_test_df = conplag_test_df[conplag_test_df["code1"].str.strip().str.len() > 5]
conplag_test_df = conplag_test_df[conplag_test_df["code2"].str.strip().str.len() > 5]

after_empty_conplag_test = len(conplag_test_df)

print(f"Removed {initial_count_conplag_test - after_empty_conplag_test} pairs with empty code")

Removed 0 pairs with empty code


In [21]:
conplag_train_df = conplag_train_df.drop_duplicates(subset=["code1", "code2"])

after_duplicates_conplag_train = len(conplag_train_df)

print(f"Removed {after_empty_conplag_train - after_duplicates_conplag_train} duplicated pairs")

Removed 6 duplicated pairs


In [22]:
conplag_test_df = conplag_test_df.drop_duplicates(subset=["code1", "code2"])

after_duplicates_conplag_test = len(conplag_test_df)

print(f"Removed {after_empty_conplag_test - after_duplicates_conplag_test} duplicated pairs")

Removed 29 duplicated pairs


In [23]:
print(f"Final pairs (train): {after_duplicates_conplag_train}")
print(f"Final pairs (test): {after_duplicates_conplag_test}")

Final pairs (train): 454
Final pairs (test): 1333


### 3.3 Verification for ai_detection dataset

In [24]:
if df_ai_detection.isnull().values.any():
    print("There are null values in the dataset, data cleaning is required.")
else:
    print("There are no null values in the AI detection dataset, data is clean.")

There are no null values in the AI detection dataset, data is clean.


In [25]:
initial_count_ai = len(df_ai_detection)

df_ai_detection = df_ai_detection[df_ai_detection["code"].str.strip().str.len() > 5]

after_empty_ai = len(df_ai_detection)

print(f"Removed {initial_count_ai - after_empty_ai} examples with empty code")

Removed 0 examples with empty code


In [26]:
df_ai_detection = df_ai_detection.drop_duplicates(subset=["code"])

after_duplicates_ai = len(df_ai_detection)

print(f"Removed {after_empty_ai - after_duplicates_ai} duplicated examples")

Removed 696 duplicated examples


In [27]:
print(f"Final pairs: {after_duplicates_ai}")

Final pairs: 442896


### 3.4 Verification for plagiarism_detection dataset

In [28]:
if df_plagiarism_detection.isnull().values.any():
    print("There are null values in the dataset, data cleaning is required.")
else:
    print("There are no null values in the plagiarism detection dataset, data is clean.")

There are no null values in the plagiarism detection dataset, data is clean.


In [29]:
initial_count_plag = len(df_plagiarism_detection)

df_plagiarism_detection = df_plagiarism_detection[df_plagiarism_detection["code1"].str.strip().str.len() > 5]
df_plagiarism_detection = df_plagiarism_detection[df_plagiarism_detection["code2"].str.strip().str.len() > 5]

after_empty_plag = len(df_plagiarism_detection)

print(f"Removed {initial_count_plag - after_empty_plag} pairs with empty code")

Removed 29 pairs with empty code


In [30]:
df_plagiarism_detection = df_plagiarism_detection.drop_duplicates(subset=["code1", "code2"])

after_duplicates_plag = len(df_plagiarism_detection)

print(f"Removed {after_empty_plag - after_duplicates_plag} duplicated pairs")

Removed 132980 duplicated pairs


In [31]:
print(f"Final pairs: {after_duplicates_plag}")

Final pairs: 416022


## 4. Data preprocessing

### 4.1. Data sampling from plagiarism datasets

In [38]:
df_ir_plag_sample = df_ir_plag[["code1", "code2", "label"]].copy()

conplag_full = pd.concat([conplag_train_df, conplag_test_df], ignore_index=True)
conplag_plag = conplag_full[conplag_full["label"] == 1]
# conplag has significantly more 0 pairs, so we downsample to the number of 1 pairs
conplag_nonplag = conplag_full[conplag_full["label"] == 0].sample(
    n=len(conplag_plag), random_state=42
)
df_conplag_sample = pd.concat([conplag_plag, conplag_nonplag], ignore_index=True)

df_id2sc_plag = df_plagiarism_detection[df_plagiarism_detection["label"] == 1].sample(
    n=14178, random_state=42
)
df_id2sc_nonplag = df_plagiarism_detection[df_plagiarism_detection["label"] == 0].sample(
    n=14415, random_state=42
)
df_id2sc_sample = pd.concat([df_id2sc_plag, df_id2sc_nonplag], ignore_index=True)[
    ["code1", "code2", "label"]
]

df_plagiarism_final = pd.concat(
    [df_ir_plag_sample, df_conplag_sample, df_id2sc_sample],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle each dataframe source

print(df_plagiarism_final["label"].value_counts())
print(f"Total pairs: {len(df_plagiarism_final)}")

label
1    15000
0    15000
Name: count, dtype: int64
Total pairs: 30000


### 4.2 Data sampling from AI detection dataset

In [39]:
df_ai_final = pd.concat([
    df_ai_detection[df_ai_detection["label"] == 0].sample(n=15000, random_state=42),
    df_ai_detection[df_ai_detection["label"] == 1].sample(n=15000, random_state=42)
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_ai_final["label"].value_counts())
print(f"Total AI pairs: {len(df_ai_final)}")

label
0    15000
1    15000
Name: count, dtype: int64
Total AI pairs: 30000


### 4.3 Data split for unified plagiarism dataset

In [40]:
# 80% train
df_plagiarism_train, df_plagiarism_rest = train_test_split(
    df_plagiarism_final, test_size=0.2, random_state=42, stratify=df_plagiarism_final["label"]
)

# 10% val / 10% test
df_plagiarism_val, df_plagiarism_test = train_test_split(
    df_plagiarism_rest, test_size=0.5, random_state=42, stratify=df_plagiarism_rest["label"]
)

print("============ TRAIN ============ ")
print(f"Train: {len(df_plagiarism_train)} code pairs")
print(df_plagiarism_train['label'].value_counts())
print("============ VAL ============ ")
print(f"Val: {len(df_plagiarism_val)} code pairs")
print(df_plagiarism_val['label'].value_counts())
print("============ TEST ============ ")
print(f"Test: {len(df_plagiarism_test)} code pairs")
print(df_plagiarism_test['label'].value_counts())

============ TRAIN ============ 
Train: 24000 code pairs
label
0    12000
1    12000
Name: count, dtype: int64
============ VAL ============ 
Val: 3000 code pairs
label
0    1500
1    1500
Name: count, dtype: int64
============ TEST ============ 
Test: 3000 code pairs
label
1    1500
0    1500
Name: count, dtype: int64


### 4.4 Data split for AI detection dataset

In [41]:
df_ai_train, df_ai_rest = train_test_split(
    df_ai_final, test_size=0.2, random_state=42, stratify=df_ai_final["label"]
)
df_ai_val, df_ai_test = train_test_split(
    df_ai_rest, test_size=0.5, random_state=42, stratify=df_ai_rest["label"]
)

print("============ TRAIN ============ ")
print(f"Train: {len(df_ai_train)} code pairs")
print(df_ai_train['label'].value_counts())
print("============ VAL ============ ")
print(f"Val: {len(df_ai_val)} code pairs")
print(df_ai_val['label'].value_counts())
print("============ TEST ============ ")
print(f"Test: {len(df_ai_test)} code pairs")
print(df_ai_test['label'].value_counts())

============ TRAIN ============ 
Train: 24000 code pairs
label
0    12000
1    12000
Name: count, dtype: int64
============ VAL ============ 
Val: 3000 code pairs
label
0    1500
1    1500
Name: count, dtype: int64
============ TEST ============ 
Test: 3000 code pairs
label
1    1500
0    1500
Name: count, dtype: int64


### 4.5 Save preprocessed dataframes in CSVs

In [42]:
df_plagiarism_train.to_csv("df_plagiarism_train.csv", index=False)
df_plagiarism_val.to_csv("df_plagiarism_val.csv", index=False)
df_plagiarism_test.to_csv("df_plagiarism_test.csv", index=False)

In [43]:
df_ai_train.to_csv("df_ai_train.csv", index=False)
df_ai_val.to_csv("df_ai_val.csv", index=False)
df_ai_test.to_csv("df_ai_test.csv", index=False)